# GridOps SFT Pipeline

This notebook verifies the original GridOps OpenEnv, loads the 1,200-row SFT curriculum, runs baseline/adversarial checks, and optionally launches compact Qwen2.5 SFT. Training cells are guarded and default to read-only.

## Links

- Space: https://77ethers-gridops.hf.space/dashboard/
- Space repo: https://huggingface.co/spaces/77ethers/gridops
- Planned model repo: https://huggingface.co/77ethers/gridops-models
- Dataset file: `sft_traces/gridops_curriculum_1200.jsonl`
- Model card: `GRIDOPS_MODEL_CARD.md`

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/capabl-machines/gridops.git'
BRANCH = 'codex/gridops-sft-pipeline'

if 'google.colab' in sys.modules:
    root = Path('/content/gridops')
    if not root.exists():
        subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(root)], check=True)
    os.chdir(root)

%pip install -q -e .
%pip install -q pytest pandas matplotlib huggingface_hub

In [ ]:
import openenv
from gridops.server.environment import GridOpsEnvironment
from gridops.models import GridOpsAction

print('OpenEnv version:', getattr(openenv, '__version__', 'unknown'))
env = GridOpsEnvironment()
obs = env.reset(seed=42, task_id='task_1_normal')
print(obs.model_dump())
obs = env.step(GridOpsAction())
print('step reward:', obs.reward, 'done:', obs.done)

## Oracle and Adversarial Smoke Checks

In [ ]:
subprocess.run([sys.executable, 'scripts/oracle_test.py'], check=True)

In [ ]:
import json, pandas as pd
from scripts.evaluate_gridops_model import evaluate_policy

policies = ['oracle', 'do_nothing', 'always_charge', 'always_discharge', 'always_diesel', 'shed_farmer', 'diesel_chatter', 'blackout_acceptor', 'price_greedy', 'grid_only']
reports = [evaluate_policy(name, [42]) for name in policies]
pd.DataFrame([{'policy': r['name'], 'average_score': r['average_score'], **r['by_task']} for r in reports])

## Curriculum Trace Validation

In [ ]:
trace_path = Path('sft_traces/gridops_curriculum_1200.jsonl')
if not trace_path.exists():
    subprocess.run([sys.executable, 'scripts/generate_sft_traces.py'], check=True)
subprocess.run([sys.executable, 'scripts/validate_traces.py', str(trace_path)], check=True)

rows = [json.loads(line) for line in trace_path.read_text().splitlines() if line.strip()]
pd.DataFrame(rows).groupby(['task_id', 'difficulty']).size().reset_index(name='rows')

In [ ]:
sample = rows[0]
print(sample['messages'][1]['content'])
print('completion:', sample['completion'])
print('focus tags:', sample['raw']['focus_tags'])

## Optional SFT Launch

This cell is off by default. It uploads to a new model subfolder and does not touch the environment Space.

In [ ]:
RUN_SFT = False

if RUN_SFT:
    assert os.environ.get('HF_API_TOKEN') or os.environ.get('HF_TOKEN'), 'Set HF_API_TOKEN first'
    subprocess.run([sys.executable, 'scripts/hf_sft_gridops.py'], check=True)
else:
    print('Skipped. Set RUN_SFT=True to launch compact GridOps SFT.')